# DAT535 Project – Group 24: Norgespris and electricity use
## Notebook 01: Scope, Dataset and Bronze layer

This notebook implements the first three sections of our *Data Engineering Project Definition Guide (v3)*:

| Section | What this notebook does |
|---|---|
| 1. Project statement | Writes down the scope we build against |
| 2. Dataset | Describes the sources, then **profiles the real files with Spark** (section 4 of this notebook) |
| 3. Landing (ingest) | Downloads / copies raw files **unchanged** into the landing folder `/tmp/norgespris-landing/<env>/`, writes a manifest with checksums, skips duplicates, flags schema changes |
| 5. Bronze layer | Loads the landing files into **PostgreSQL**, schema `bronze` (one table per dataset) |

**How this notebook fits the repo**

* The same ingestion code is in `bronze_pipeline.py` (generated from the code cells in sections 3.1–3.9), so GitHub Actions can
  run it without Jupyter: `python run_pipeline.py bronze`.
* Dev and prod are separated by two environment variables, set by the workflows in `.github/workflows/`:
  `NORGESPRIS_ENV` (`dev`/`prod`) – it picks the landing folder (`/tmp/norgespris-landing/dev` or `/prod`) and the database (`norgespris_dev` / `norgespris_prod`).
  When you run the notebook yourself it defaults to **dev**.
* Credentials (Frost, PostgreSQL) are read from the file **`.env`** in the repo root – never written in the notebook, never committed to git.
* Data flow: sources → **landing** (raw files in `/tmp/norgespris-landing/<env>/`, section 3) → **Bronze** (tables `bronze.*` in PostgreSQL, section 5, code also in `bronze_to_postgres.py`).
* Tests for the Bronze code: `pytest -q tests/`.

---
# 1. Project statement (scope)

**What we build:** a batch pipeline in PySpark with Bronze, Silver and Gold layers that reads data from
Elhub (electricity), MET/Frost (temperature) and SSB (population).

**Question it answers:** *Has daily electricity use per inhabitant in each municipality changed after Norgespris
started (1 October 2025), when we take temperature into account – and in which municipalities is the change largest?*

**Result:** clean Silver tables, a Gold table at grain *municipality × day*, and a simple dashboard with
municipal time series and a before/after comparison.

**Users:** our group and the examiner; possibly analysts interested in the same question.

**Out of scope (on purpose):** single households (EV charging, heat pumps, wood stoves), electricity prices,
real-time/streaming processing.

**Why this scope:** only open data, a clear before/after date, and real Spark work
(hourly → daily aggregation of ~15 million rows per year, joins on municipality and date) that still fits our machines.

**Analysis period:** from **November 2023** (the oldest data Elhub's API still serves – see section 5) to the last complete month.
That gives two winters before Norgespris (2023/24 and 2024/25) and at least one after.

---
# 2. Dataset (source and suitability)

| # | Dataset (folder in landing = table in `bronze`) | From | Format | One row / record is | How we get it |
|---|---|---|---|---|---|
| 1 | `consumption_per_group_municipality_hour` | Elhub Energy Data API | ZIP with CSV | kWh and number of metering points per **municipality × hour × consumption group** | API CSV export: last ~3 years in one file |
| 2 | `norgespris_consumption_per_group_eac_mba` | Elhub Energy Data API | ZIP with CSV | kWh **with / without** Norgespris per **day × price area × group × annual-consumption band** | API CSV export |
| 3 | `norgespris_count_per_municipality_consumption_group` | Elhub open data (data.elhub.no) | CSV (gzip) | Number of metering points **with Norgespris** per municipality and group | Download, full history (*optional*) |
| 4 | `population_07459` | SSB table 07459 | CSV `;` | Population 1 January per **municipality** (one column per year) | SSB PxWebApi v2 |
| 5 | `frost_stations` | MET Frost API `/sources` | JSON | One weather **station** with its **municipality** (`municipalityId`) and county | API, one file |
| 6 | `frost_timeseries` | MET Frost API `/observations/availableTimeSeries` | JSON | One station that **has** daily mean temperature, with from/to dates | API, one file |
| 7 | `frost_observations` | MET Frost API `/observations` | JSON | Daily mean air temperature per **station × day** | API, one file per month per 100 stations |
| 8 | `seklima_daily_mean_temperature` | MET Seklima (manual export) | CSV `;` | Daily mean temperature per station × day | *Optional fallback* if Frost is down |

**How we get mean temperature per municipality** (the Bronze layer only collects the raw data; the calculation is Silver work – previewed in 4.5):
1. `frost_stations` tells which **municipality** each station is in.
2. `frost_timeseries` tells which stations actually **measure** `mean(air_temperature P1D)` – so we do not ask for data from rain-only or wind-only stations.
3. `frost_observations` gives the daily mean per station.
4. Silver: join 3 with 1 on station id, then **average all stations in a municipality per day**.

**Why it suits distributed processing:** dataset 1 alone is ~357 municipalities × 8 760 hours × ~5 groups ≈ 15 million rows
per year. Two–three years must be aggregated to days and joined with population, temperature and Norgespris data –
a typical Spark job (shuffle on `groupBy`, joins on municipality and date).

**Licences:** Elhub open data (NLOD), SSB (CC BY 4.0), MET Norway / Frost (CC BY 4.0).

**Reproducibility:** every file is kept unchanged in the landing folder with source URL, time, size and SHA-256 checksum (section 3),
and loaded as raw text into the `bronze` schema in PostgreSQL (section 5) with the file name and run on every row.
Elhub's API only serves the last ~3 years, so **old months disappear from the source** – keep that in mind: the landing folder
is in `/tmp` and is cleaned by the system, and the Bronze table holds the newest export. Months that matter long-term are kept in Silver.

**Known quality risks** (checked with Spark in section 4): different municipality code formats, municipality `0` = "unknown",
broken Norwegian letters in some files, municipalities without a weather station, several daily-mean series per station
(different `timeOffset`), time zones (Elhub `+01:00/+02:00`, Frost UTC), revised Elhub data after month end, only yearly population.

---
# 3. Landing (ingest raw files)

**Design choices, and why:**

| Choice | Why |
|---|---|
| Keep files **as published** in the landing folder | The landing files are the proof of what the source delivered. All cleaning is done in Silver, where it can be tested and rerun. (Only gzip *transfer* compression is undone – see 3.6.) |
| Plain Python (`urllib`, `hashlib`, `csv`, `json`) for ingestion; Spark only for reading/checking | Downloading files is not a distributed task. Using only the standard library means nothing extra to install. Spark is used where it adds value (section 4 and Silver). |
| Credentials in **`.env`**, read at start | Keeps the Frost client ID out of the code and out of git (`.env` is in `.gitignore`). GitHub Actions gets the same value from a repository secret. |
| Folder layout `/tmp/norgespris-landing/<env>/<source>/<dataset>/<period>/<run_id>__<file>` | Source and dataset visible in the path; `period` is the month (`2025-10`), `snapshot` or `manual`; `run_id` (UTC time of the run) shows *when* it was ingested and keeps old versions side by side. |
| One **manifest** file `/tmp/norgespris-landing/<env>/_manifest.csv` | One line per file: run, source URL, file name, time, size, SHA-256, header line, schema status, status. Our traceability *and* monitoring log. |
| **Skip** a file whose checksum already exists for the dataset | Reruns are safe (idempotent) and never create duplicates. |
| Frost observations: **one request per month per 100 stations** | Keeps each answer small (≈ 3 000 values), well below Frost's limits, and gives the year/month folder structure from the definition. Months already stored are not requested again; the **2 newest months** are always re-requested because MET corrects recent values. |
| Compare the **header line** with the previous file of the dataset | A changed CSV header is flagged (`schema_status = changed`) but the file is kept, so Silver decides. (Not used for JSON.) |
| `required` flag per source | A failing **required** source makes the run fail (red in GitHub Actions). Optional sources only give a warning. |
| Landing folder in **`/tmp`** (`/tmp/norgespris-landing/<env>/`) | Raw files are short-lived input: once loaded into the `bronze` schema in PostgreSQL, the database holds the data. `/tmp` is cleaned by the system, so the disk does not fill up with 3-year Elhub exports. Downloads arrive in `_incoming/` first (only complete files are kept); manual files go in `_manual/<source>/<dataset>/`. |
| Period in **`.env`** (dev = 2 months, prod = Nov 2023 → last complete month) | Small dev runs are quick; prod gets the full analysis period. Changing the period is a config change, not a code change. |

### 3.1 Imports
Only the Python standard library is needed for ingestion.

In [1]:
import base64                                  # build the HTTP Basic-auth header for the Frost API
import csv                                     # read and write the manifest file (handles commas/quotes in text)
import gzip                                    # data.elhub.no sends files gzip-compressed
import hashlib                                 # SHA-256 checksums to recognise identical files
import json                                    # read the Frost station list to know which stations to ask for
import logging                                 # log messages that also show up in GitHub Actions
import os                                      # environment variables and file paths
import shutil                                  # copy files and delete temp folders
import time                                    # measure run time and wait between download retries
import urllib.error                            # tells us the HTTP status code when a request fails (e.g. 404)
import urllib.request                          # download files over HTTPS (standard library, no pip install needed)
import zipfile                                 # Elhub's ".csv" exports are really ZIP files with a CSV inside
from datetime import datetime, timezone, date  # timestamps (always UTC) and month calculations

logging.basicConfig(level=logging.INFO,                                        # show INFO and above
                    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")  # time - logger - level - text
logger = logging.getLogger("bronze")                                          # named logger for this layer

### 3.2 Configuration: credentials, environment and folders

**Credentials:** `load_env_file()` reads `KEY=value` lines from `.env` (repo root) into environment variables.
It looks in the current folder and the parent folder, because the notebook runs from `notebooks/` and the script from the repo root.
Values that are already set (e.g. by GitHub Actions secrets) are **not** overwritten. We only print *which* keys were found – never the values.

**Environment:** `NORGESPRIS_ENV` is set by the workflows (`dev`/`prod`). When it is missing (you run the notebook yourself) we use **dev**.

**Landing folder** `/tmp/norgespris-landing/<env>/` (change it with `NORGESPRIS_LANDING_ROOT` in `.env`):

```text
/tmp/norgespris-landing/dev/
├── _manifest.csv                              one line per file: source, time, size, SHA-256, status
├── _incoming/<run_id>/                        downloads arrive here; moved on when complete, then deleted
├── _manual/<source>/<dataset>/                drop manual exports here (e.g. a Seklima CSV)
└── <source>/<dataset>/<period>/<run_id>__<file>   the stored raw files, e.g.
    elhub/consumption_per_group_municipality_hour/snapshot/20261111T...Z__consumption_per_group_municipality_hour.zip
```

Why `/tmp`: landing is short-lived input. After section 5 the data is in the `bronze` schema in PostgreSQL.
`/tmp` is cleaned by the operating system (on reboot, and on macOS also files not used for a few days). Then the next run simply
downloads again (the manifest is gone too, so nothing is skipped) – correct, only slower. Set `NORGESPRIS_LANDING_ROOT` to a folder
in your home directory if you want the raw files to survive.

**Period (which months to fetch from Frost)** is *configuration*, not code, so it lives in `.env`:

| Key in `.env` | Example | Meaning |
|---|---|---|
| `NORGESPRIS_DEV_START_MONTH` | `2025-09` | first month in dev |
| `NORGESPRIS_DEV_END_MONTH` | `2025-10` | last month in dev |
| `NORGESPRIS_PROD_START_MONTH` | `2023-11` | first month in prod |
| `NORGESPRIS_PROD_END_MONTH` | *(empty)* | last month in prod – empty = last complete month |

Why: the group can change the period without touching (and re-testing) the code, and dev and prod read the same code with different values.
`month_range()` reads the two keys that belong to the current environment and stops with a clear message if the start month is missing or not written as `YYYY-MM`.
It is called when the Frost requests are built (3.8), not when the code is loaded, so tests and the other sources still work without it.

In [2]:
def load_env_file(candidates=(".env", os.path.join("..", ".env"))):
    """Read KEY=value lines from the first .env file found into os.environ (does not overwrite existing values)."""
    for path in candidates:                                   # repo root seen from the root or from notebooks/
        if not os.path.exists(path):
            continue
        loaded = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()                           # remove spaces and the line break
                if not line or line.startswith("#") or "=" not in line:  # skip empty lines and comments
                    continue
                key, value = line.split("=", 1)               # split at the FIRST "=" only
                key = key.replace("export ", "").strip()      # allow "export KEY=value" too
                value = value.strip().strip('"').strip("'")   # remove spaces and surrounding quotes
                if key and key not in os.environ:             # real environment variables win (GitHub secrets)
                    os.environ[key] = value
                    loaded.append(key)
        logger.info(f"Loaded {loaded or 'no new keys'} from {os.path.abspath(path)}")  # names only, never values
        return
    logger.info("No .env file found - using environment variables only")

load_env_file()                                               # run it once when the code is loaded

ENV = os.environ.get("NORGESPRIS_ENV", "dev")                 # "dev" or "prod"; default dev when run by hand

LANDING_ROOT = os.environ.get("NORGESPRIS_LANDING_ROOT",      # where the raw files are stored ...
                              "/tmp/norgespris-landing")      # ... default: a folder in /tmp
LANDING_DIR = os.path.join(LANDING_ROOT, ENV)                 # /tmp/norgespris-landing/dev or .../prod
MANUAL_DIR = os.path.join(LANDING_DIR, "_manual")             # drop manually exported files here
MANIFEST_PATH = os.path.join(LANDING_DIR, "_manifest.csv")    # one line per file we have seen

REDOWNLOAD_RECENT_MONTHS = 2                                  # always re-fetch the newest months (MET corrects recent data)
FROST_STATIONS_PER_REQUEST = 100                              # stations per observations request (keeps answers small)


def month_range():
    """(start, end) months for the current environment from .env; end is None = up to the last complete month."""
    prefix = f"NORGESPRIS_{ENV.upper()}"                       # NORGESPRIS_DEV or NORGESPRIS_PROD
    start = os.environ.get(f"{prefix}_START_MONTH", "").strip()  # e.g. "2025-09"
    end = os.environ.get(f"{prefix}_END_MONTH", "").strip() or None  # empty -> None
    for key, value in ((f"{prefix}_START_MONTH", start), (f"{prefix}_END_MONTH", end)):
        if value is None and key.endswith("END_MONTH"):        # an empty end month is allowed
            continue
        if len(value) != 7 or value[4] != "-" or not (value[:4] + value[5:]).isdigit():  # must look like YYYY-MM
            raise ValueError(f"{key} must be set as YYYY-MM in .env (or as environment variable), got '{value}'")
        try:
            date.fromisoformat(f"{value}-01")
        except ValueError as e:
            raise ValueError(f"{key} must be a valid calendar month (YYYY-MM), got '{value}'") from e
    return start, end

2026-11-13 00:38:31,107 - bronze - INFO - Loaded ['FROST_CLIENT_ID', 'FROST_CLIENT_SECRET', 'NORGESPRIS_DEV_START_MONTH', 'NORGESPRIS_DEV_END_MONTH', 'NORGESPRIS_PROD_START_MONTH', 'NORGESPRIS_PROD_END_MONTH', 'NORGESPRIS_LANDING_ROOT', 'NORGESPRIS_PG_HOST', 'NORGESPRIS_PG_PORT', 'NORGESPRIS_PG_USER', 'NORGESPRIS_PG_PASSWORD'] from /Users/x/UiS/DAT535 Data Engineering/dat535-group24/.env


### 3.3 Source registry
One dictionary per dataset – **adding a dataset = adding one entry here.**

* `snapshot` – one file with everything, fetched every run (unchanged content is skipped by checksum).
  Elhub's `.csv` export accepts **no** `startDate`/`endDate` (HTTP 400), so it always returns the last ~3 years in one file.
* `frost_observations` – many requests: every month × every group of 100 stations (built in 3.8).
* `manual` – no download; only files placed in `landing/<source>/<dataset>/`.
* `sep` – column separator for CSV files (`None` = JSON).
* `required` – if `False`, a failed download is only a warning.
* `auth` – `"frost"` = send the Frost client ID.

**The three Frost calls** (the first two are the calls from our project description, the third is added by us):

| Dataset | URL | Why |
|---|---|---|
| `frost_stations` | `/sources/v0.jsonld?types=SensorSystem&validtime=2023-01-01/2027-01-01` | every station that existed in our period, **with its municipality** |
| `frost_timeseries` | `/observations/availableTimeSeries/v0.jsonld?elements=mean(air_temperature P1D)&referencetime=2023-11-01/2027-01-01` | only the stations that **have** daily mean temperature – about a third of all stations, so far fewer requests |
| `frost_observations` | `/observations/v0.jsonld?elements=mean(air_temperature P1D)&sources=SN…,SN…&referencetime=<month>` | the actual daily means |

The space in `mean(air_temperature P1D)` is written `%20` in a URL.

Every dataset **also** picks up files from its landing folder `/tmp/norgespris-landing/<env>/<source>/<dataset>/`.

In [3]:
ELHUB_API = "https://api.elhub.no/energy-data/v0"             # Elhub Energy Data API (open, no login)
FROST_API = "https://frost.met.no"                            # MET Norway's Frost API (needs a client ID)
FROST_ELEMENT = "mean(air_temperature%20P1D)"                 # daily mean air temperature (%20 = space)

SSB_URL = ("https://data.ssb.no/api/pxwebapi/v2/tables/07459/data?lang=no"      # SSB table 07459, Norwegian labels
           "&valueCodes[Region]=*&codelist[Region]=agg_KommSummer"             # all municipalities, today's structure
           "&outputValues[Region]=aggregated"                                  # sums for merged municipalities
           "&valueCodes[ContentsCode]=*&valueCodes[Tid]=from(2023)"            # population, years 2023 ->
           "&outputFormat=csv&outputFormatParams=separatorsemicolon,usecodesandtexts")  # ';' + code AND name ('3101 Halden')
# Sex and age are left out of the query -> SSB sums them ("elimination"), so we get total population.

SOURCES = [                                                                    # the datasets the pipeline handles
    {"source": "elhub", "dataset": "consumption_per_group_municipality_hour",  # hourly kWh per municipality and group
     "mode": "snapshot", "sep": ";", "required": True,                         # .csv export = last ~3 years, no date filter
     "url": ELHUB_API + "/municipalities.csv?dataset=CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR"},
    {"source": "elhub", "dataset": "norgespris_consumption_per_group_eac_mba", # kWh with/without Norgespris per price area
     "mode": "snapshot", "sep": ",", "required": False,
     "url": ELHUB_API + "/price-areas.csv?dataset=NORGESPRIS_CONSUMPTION_PER_GROUP_EAC_MBA"},
    {"source": "elhub", "dataset": "norgespris_count_per_municipality_consumption_group",  # Norgespris meters per municipality
     "mode": "snapshot", "sep": ",", "required": False,
     "url": "https://data.elhub.no/download/norgespris_count_per_municipality_consumption_group/"
            "norgespris_count_per_municipality_consumption_group-all-en-0000-00-00.csv"},
    {"source": "ssb", "dataset": "population_07459",                          # population per municipality
     "mode": "snapshot", "sep": ";", "required": True, "url": SSB_URL},
    {"source": "met", "dataset": "frost_stations",                            # 1) station -> municipality
     "mode": "snapshot", "sep": None, "required": True, "auth": "frost", "ext": "json",
     "url": FROST_API + "/sources/v0.jsonld?types=SensorSystem&validtime=2023-01-01/2027-01-01"},
    {"source": "met", "dataset": "frost_timeseries",                          # 2) which stations have daily mean temp
     "mode": "snapshot", "sep": None, "required": True, "auth": "frost", "ext": "json",
     "url": FROST_API + "/observations/availableTimeSeries/v0.jsonld?elements=" + FROST_ELEMENT
                      + "&referencetime=2023-11-01/2027-01-01"},
    {"source": "met", "dataset": "frost_observations",                        # 3) daily mean temp per station
     "mode": "frost_observations", "sep": None, "required": True, "auth": "frost", "ext": "json",
     "url": FROST_API + "/observations/v0.jsonld?elements=" + FROST_ELEMENT
                      + "&sources={sources}&referencetime={start}/{end}"},
    {"source": "met", "dataset": "seklima_daily_mean_temperature",            # manual fallback, not needed with Frost
     "mode": "manual", "sep": ";", "required": False},
]

MANIFEST_COLUMNS = ["run_id", "env", "source", "dataset", "period", "origin",  # columns of _manifest.csv
                    "original_filename", "landing_path", "ingested_at_utc", "size_bytes",
                    "sha256", "header", "schema_status", "status"]

### 3.4 Helper functions: checksum, header and manifest
* **SHA-256** is a fingerprint of the content: same content → same checksum. Reading in 1 MB blocks means a big file never has to fit in memory.
* **ZIP files:** Elhub's `.csv` export is a ZIP containing Spark-style `part-00000-….csv` files. A ZIP stores timestamps, so its bytes change on
  every download even when the data is the same. For ZIPs we therefore checksum the **files inside**, and read the header from the first CSV inside.
* The **header line** is stored in the manifest to detect schema changes in CSV files. For JSON (Frost) there is no header line, so we store `(json)`.
* The **manifest** is only appended to, never rewritten, so it holds the history of all runs.

In [4]:
def sha256_of(path):
    """Return the SHA-256 checksum of a file."""
    h = hashlib.sha256()                                       # empty checksum object
    with open(path, "rb") as f:                                # binary mode: we hash the exact bytes
        for block in iter(lambda: f.read(1024 * 1024), b""):   # read 1 MB at a time until end of file
            h.update(block)                                    # add the block to the checksum
    return h.hexdigest()                                       # 64-character hex string


def content_sha256(path):
    """Checksum of the data: for a ZIP the files inside (in name order), otherwise the file itself."""
    if not zipfile.is_zipfile(path):                           # normal file (CSV/JSON)
        return sha256_of(path)
    h = hashlib.sha256()
    with zipfile.ZipFile(path) as zf:                          # open the archive
        for name in sorted(zf.namelist()):                     # same order every time
            with zf.open(name) as member:                      # read one file inside the ZIP
                for block in iter(lambda: member.read(1024 * 1024), b""):  # 1 MB at a time
                    h.update(block)
    return h.hexdigest()


def read_header(path):
    """First line of a CSV (or of the first CSV inside a ZIP), without BOM and line break; '(json)' for JSON."""
    if path.lower().endswith(".json"):                         # JSON has no header line to compare
        return "(json)"
    if zipfile.is_zipfile(path):                               # ZIP -> look at the first CSV inside
        with zipfile.ZipFile(path) as zf:
            csvs = sorted(n for n in zf.namelist() if n.lower().endswith(".csv"))  # CSV members only
            if not csvs:                                       # a ZIP without CSV: nothing to compare
                return ""
            with zf.open(csvs[0]) as member:
                first = member.readline(2000)                  # first line of the first CSV
    else:
        with open(path, "rb") as f:                            # binary mode, so odd characters cannot crash us
            first = f.readline(2000)                           # first line only, max 2000 bytes
    return first.decode("utf-8-sig", errors="replace").strip() # utf-8-sig removes the BOM, strip removes \r\n


def read_manifest():
    """Return all rows of the manifest as a list of dicts (empty list on the first run)."""
    if not os.path.exists(MANIFEST_PATH):                      # first run: no manifest yet
        return []
    with open(MANIFEST_PATH, newline="", encoding="utf-8") as f:  # newline="" is required by the csv module
        return list(csv.DictReader(f))                         # one dict per row, keys = column names


def append_manifest(row):
    """Append one row to the manifest (writes the column names if the file is new)."""
    os.makedirs(LANDING_DIR, exist_ok=True)                    # make sure the landing folder exists
    is_new = not os.path.exists(MANIFEST_PATH)                 # do we need to write the column names?
    with open(MANIFEST_PATH, "a", newline="", encoding="utf-8") as f:  # "a" = append, never overwrite
        writer = csv.DictWriter(f, fieldnames=MANIFEST_COLUMNS, restval="")  # missing keys become ""
        if is_new:
            writer.writeheader()                               # column names first in a new manifest
        writer.writerow(row)                                   # this file's metadata

### 3.5 Month helpers
Frost observations are requested month by month. The newest month we ask for in prod is the **last complete month**.

In [5]:
def month_list(start, end):
    """List of 'YYYY-MM' strings from start to end, both included."""
    y, m = map(int, start.split("-"))                          # start year and month as numbers
    ey, em = map(int, end.split("-"))                          # end year and month as numbers
    months = []
    while (y, m) <= (ey, em):                                  # tuples compare year first, then month
        months.append(f"{y:04d}-{m:02d}")                      # e.g. "2025-09"
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)           # next month (December -> January next year)
    return months


def next_month(month):
    """'2025-12' -> '2026-01'."""
    y, m = map(int, month.split("-"))                          # split into numbers
    y, m = (y + 1, 1) if m == 12 else (y, m + 1)               # one month forward
    return f"{y:04d}-{m:02d}"


def last_complete_month(today=None):
    """The month before the current one, e.g. '2026-10' in November 2026."""
    today = today or date.today()                              # allow a fixed date in tests
    y, m = (today.year - 1, 12) if today.month == 1 else (today.year, today.month - 1)  # one month back
    return f"{y:04d}-{m:02d}"

### 3.6 Download: login, retries and gzip

* **Frost login:** HTTP *Basic authentication* with the **client ID as user name and an empty password** – that is what Frost's documentation asks for when reading open data.
  The client **secret** is only needed for OAuth2 (restricted data), so it is stored in `.env` but not sent.
* **Retries:** servers sometimes fail for a moment, so we try 3 times with a growing pause. But **not** for answers like 400/401/403/404 – those mean the request itself is wrong (or there is no data), and asking again will not help.
* **gzip:** `data.elhub.no` sends files gzip-compressed (the file starts with the bytes `1f 8b`). A browser unpacks this automatically, Python's `urllib` does not, so we unpack it. Gzip is only *packing for the transfer*; ZIP files are what Elhub actually publishes, so those are kept as they are (3.7).

In [6]:
def gunzip_if_needed(path):
    """If the file is gzip-compressed, replace it with the unpacked content."""
    with open(path, "rb") as f:
        if f.read(2) != b"\x1f\x8b":                           # gzip files always start with these two bytes
            return                                             # not gzip -> nothing to do
    with gzip.open(path, "rb") as src, open(path + ".unpacked", "wb") as dst:  # read compressed, write plain
        shutil.copyfileobj(src, dst)                           # stream, so big files are fine
    os.replace(path + ".unpacked", path)                       # swap the unpacked file in
    logger.info(f"  unpacked gzip: {os.path.basename(path)}")


def auth_headers(source_cfg):
    """Extra HTTP headers for sources that need a login. {} = open source, None = login missing."""
    if source_cfg.get("auth") != "frost":                      # Elhub and SSB are open
        return {}
    client_id = os.environ.get("FROST_CLIENT_ID", "").strip()  # from .env or a GitHub secret
    if not client_id or client_id.startswith("your-"):         # missing, or still the placeholder from .env
        return None
    token = base64.b64encode(f"{client_id}:".encode()).decode()  # "<client id>:" = empty password (Frost docs)
    return {"Authorization": f"Basic {token}"}


def download(url, target_path, headers=None, attempts=3):
    """Download url to target_path; retry server/network errors, raise the last error if all attempts fail."""
    all_headers = {"User-Agent": "DAT535-group24-student-project"}  # say politely who we are
    all_headers.update(headers or {})                          # add e.g. the Authorization header
    for attempt in range(1, attempts + 1):                     # attempt 1, 2, 3
        try:
            req = urllib.request.Request(url, headers=all_headers)                    # build the request
            with urllib.request.urlopen(req, timeout=300) as resp, open(target_path, "wb") as out:  # max 5 min
                shutil.copyfileobj(resp, out)                  # stream the response to the file
            gunzip_if_needed(target_path)                      # undo gzip transfer compression (data.elhub.no)
            return                                             # success -> stop retrying
        except urllib.error.HTTPError as e:                    # the server answered with an error code
            if e.code in (400, 401, 403, 404):                 # our request is wrong / no access / no data
                raise                                          # -> retrying will not help
            error = e                                          # 429, 500, 503 ... -> worth another try
        except Exception as e:                                 # network problem, timeout ...
            error = e
        logger.warning(f"Download attempt {attempt}/{attempts} failed for {url}: {error}")
        if attempt == attempts:                                # the last attempt also failed ...
            raise error                                        # ... let the caller record the failure
        time.sleep(5 * attempt)                                # wait 5 s, then 10 s

### 3.7 Store one file in the landing folder
The heart of the Bronze layer. For each file:
1. compute checksum and read the header,
2. **duplicate?** same checksum already stored for this dataset → record `skipped_duplicate` and stop,
3. **schema check** (CSV only) against the newest earlier file of the dataset → `first_file`, `ok` or `changed`,
4. copy the file unchanged to `<landing>/<source>/<dataset>/<period>/<run_id>__<filename>`;
   if it is a **ZIP**, it is saved with the `.zip` ending and *also* unpacked into `<run_id>__<name>_unzipped/` next to it
   (Spark cannot read ZIP files; the ZIP stays the untouched original),
5. write one manifest line.

`state` is a small dict with the run ID, all manifest rows so far and counters for this run.

In [7]:
def ingest_file(state, src_path, source, dataset, period, origin, original_filename):
    """Store one file unchanged in the landing folder unless an identical file is already there."""
    checksum = content_sha256(src_path)                         # fingerprint of the content (inside a ZIP: the files)
    header = read_header(src_path)                              # column-names line, or "(json)"
    earlier = [r for r in state["manifest"]                     # earlier files of the SAME dataset ...
               if r["dataset"] == dataset and r["status"] == "ingested"]  # ... that were actually stored

    row = {"run_id": state["run_id"], "env": ENV, "source": source, "dataset": dataset,  # metadata for the manifest
           "period": period, "origin": origin, "original_filename": original_filename,
           "ingested_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "size_bytes": os.path.getsize(src_path), "sha256": checksum, "header": header}

    if any(r["sha256"] == checksum for r in earlier):           # exact same content seen before?
        row["landing_path"] = next(r["landing_path"] for r in reversed(earlier) if r["sha256"] == checksum)
        row["status"] = "skipped_duplicate"                     # yes -> do not store it again
        state["counts"]["skipped_duplicate"] += 1
        logger.info(f"  skip (already ingested): {dataset} {period} {original_filename}")
    else:
        if not earlier:                                         # first file ever for this dataset
            row["schema_status"] = "first_file"
        elif earlier[-1]["header"] == header:                   # same columns as the newest earlier file
            row["schema_status"] = "ok"
        else:                                                   # columns changed -> keep the file, but flag it
            row["schema_status"] = "changed"
            logger.warning(f"  HEADER CHANGED for {dataset}: check before Silver!")
        is_zip = zipfile.is_zipfile(src_path)                                 # Elhub exports are ZIPs
        stored_name = original_filename
        if is_zip and not stored_name.lower().endswith(".zip"):               # give it the correct file ending
            stored_name = os.path.splitext(stored_name)[0] + ".zip"
        target_dir = os.path.join(LANDING_DIR, source, dataset, period)       # e.g. <landing>/met/frost_observations/2025-10
        os.makedirs(target_dir, exist_ok=True)                                # create the folder if missing
        target = os.path.join(target_dir, f"{state['run_id']}__{stored_name}")  # run_id keeps versions apart
        shutil.copy2(src_path, target)                                        # byte-for-byte copy of the original
        if is_zip:                                                            # unpack a copy for Spark
            with zipfile.ZipFile(target) as zf:
                zf.extractall(target[:-len(".zip")] + "_unzipped")            # <run_id>__<name>_unzipped/part-....csv
        row["landing_path"] = os.path.relpath(target, LANDING_DIR)            # path relative to the landing folder
        row["status"] = "ingested"
        state["counts"]["ingested"] += 1
        logger.info(f"  ingested: {row['landing_path']}")

    append_manifest(row)                                        # write the line to disk right away
    state["manifest"].append(row)                               # and remember it for the rest of the run

### 3.8 Download → ingest, and the Frost observation requests

**`download_and_ingest`** downloads to a temporary file first, so only complete files are ingested. Every outcome ends up in the manifest:

| Outcome | Manifest status | Effect on the run |
|---|---|---|
| file stored / identical file already there | `ingested` / `skipped_duplicate` | – |
| Frost answers **404** for a month and group of stations | `no_data` | none – Frost says "404" when the stations have no values in that period |
| login missing (no `FROST_CLIENT_ID`) | `failed: …` / `warning: …` | required source → run **fails**, with a clear message |
| any other error | `failed: …` / `warning: …` | required source → run **fails** |

**`frost_observation_requests`** builds the list of observation requests:
1. read the newest `frost_timeseries` file from Bronze → the station ids that have daily mean temperature (`SN18700:0` → `SN18700`),
2. sort them and cut them into groups of 100,
3. for every month in the period and every group: one URL and a file name like `frost_obs_2025-10_l<layout>_g03.json`,
4. the layout signature ties filenames to the full station membership, so a changed list triggers new requests,
5. leave out requests whose file we already have – except for the 2 newest months.

In [8]:
def download_and_ingest(state, cfg, url, period, filename):
    """Download url to a temp file and ingest it; record every outcome in the manifest."""
    source, dataset = cfg["source"], cfg["dataset"]
    level = "failed" if cfg["required"] else "warning"           # how bad a failure is for this source
    base_row = {"run_id": state["run_id"], "env": ENV, "source": source, "dataset": dataset,
                "period": period, "origin": url, "original_filename": filename,
                "ingested_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    headers = auth_headers(cfg)                                  # {} for open sources, None if login is missing
    if headers is None:                                          # Frost without a client ID
        state["counts"][level] += 1
        append_manifest(dict(base_row, status=f"{level}: FROST_CLIENT_ID missing (set it in .env)"))
        logger.error(f"  {level.upper()}: {dataset} needs FROST_CLIENT_ID (in .env or as GitHub secret)")
        return
    tmp_path = os.path.join(state["tmp_dir"], filename)          # arrives in <landing>/_incoming/<run_id>/ first
    try:
        download(url, tmp_path, headers)                         # may raise after failed attempts
        ingest_file(state, tmp_path, source, dataset, period, url, filename)  # checks + copy + manifest
    except urllib.error.HTTPError as e:
        if e.code == 404 and cfg["mode"] == "frost_observations":  # Frost: "no data for these stations/month"
            state["counts"]["no_data"] += 1
            row = dict(base_row, status="no_data")
            append_manifest(row)
            state["manifest"].append(row)                        # remember: do not ask again (unless recent)
            logger.info(f"  no data: {filename}")
        else:
            state["counts"][level] += 1
            append_manifest(dict(base_row, status=f"{level}: HTTP {e.code} {e.reason}"[:200]))
            logger.error(f"  {level.upper()}: {url} (HTTP {e.code} {e.reason})")
    except Exception as e:                                       # network error etc.
        state["counts"][level] += 1
        append_manifest(dict(base_row, status=f"{level}: {type(e).__name__} {e}"[:200]))
        logger.error(f"  {level.upper()}: {url} ({e})")
    finally:
        if os.path.exists(tmp_path):                             # always clean up the temp file
            os.remove(tmp_path)


def frost_station_ids(state):
    """Station ids that have daily mean temperature, from the newest frost_timeseries file in landing."""
    rows = [r for r in state["manifest"]                         # stored availableTimeSeries files
            if r["dataset"] == "frost_timeseries"
            and r["status"] in ("ingested", "skipped_duplicate") and r.get("landing_path")]
    if not rows:                                                 # step 2 did not succeed (yet)
        return []
    path = os.path.join(LANDING_DIR, rows[-1]["landing_path"])     # newest version
    with open(path, encoding="utf-8") as f:
        data = json.load(f).get("data", [])                      # Frost puts the records in "data"
    return sorted({d["sourceId"].split(":")[0] for d in data})   # "SN18700:0" -> "SN18700", no duplicates


def frost_observation_requests(state, cfg):
    """List of (url, month, filename): every month x every group of stations, minus what we already have."""
    ids = frost_station_ids(state)                               # stations to ask for
    if not ids:
        logger.error("  no station list from frost_timeseries - cannot request observations")
        return []
    n = FROST_STATIONS_PER_REQUEST
    groups = [ids[i:i + n] for i in range(0, len(ids), n)]       # [[SN..100 ids], [SN..], ...]
    layout = hashlib.sha256(",".join(ids).encode("utf-8")).hexdigest()[:12]
    start, end = month_range()                                   # period for dev or prod, from .env
    months = month_list(start, end or last_complete_month())     # None = up to the last complete month
    recent = set(months[-REDOWNLOAD_RECENT_MONTHS:])             # always re-request the newest months
    have = {r["original_filename"] for r in state["manifest"]    # files we already got (or know are empty)
            if r["dataset"] == cfg["dataset"] and r["status"] in ("ingested", "skipped_duplicate", "no_data")}
    requests = []
    for month in months:
        for g, group in enumerate(groups, start=1):
            filename = f"frost_obs_{month}_l{layout}_g{g:02d}.json"
            if filename in have and month not in recent:         # already stored and not recent -> skip
                continue
            url = cfg["url"].format(sources=",".join(group),     # SN18700,SN18950,...
                                    start=f"{month}-01",          # first day of the month
                                    end=f"{next_month(month)}-01")  # first day of next month (end not included)
            requests.append((url, month, filename))
    logger.info(f"  {len(ids)} stations in {len(groups)} groups -> {len(requests)} requests")
    return requests

### 3.9 The whole landing run
Loops over the source registry **in order** – the order matters: `frost_timeseries` must be stored before `frost_observations` can use it.
Returns `status` and `elapsed_time`; `run_pipeline.py` turns `failed` into exit code 1 (red run in GitHub Actions).

In [9]:
def run_bronze():
    """Run Bronze ingestion for all sources and return a summary dict."""
    start_time = time.time()
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")  # microseconds keep rapid runs apart
    state = {"run_id": run_id,                                 # shared state for the helper functions
             "tmp_dir": os.path.join(LANDING_DIR, "_incoming", run_id),  # downloads arrive here first
             "manifest": read_manifest(),                      # everything ingested in earlier runs
             "counts": {"ingested": 0, "skipped_duplicate": 0, "no_data": 0, "warning": 0, "failed": 0}}
    os.makedirs(state["tmp_dir"], exist_ok=True)
    logger.info(f"Landing run {run_id} (env={ENV}) -> {LANDING_DIR}")

    for cfg in SOURCES:                                        # one dataset at a time, in registry order
        source, dataset, mode = cfg["source"], cfg["dataset"], cfg["mode"]
        logger.info(f"[{source}/{dataset}] mode={mode}")
        ext = cfg.get("ext", "csv")                            # file extension (Frost gives JSON)

        if mode == "snapshot":                                 # one file with everything
            download_and_ingest(state, cfg, cfg["url"], "snapshot", f"{dataset}.{ext}")
        elif mode == "frost_observations":                     # many small requests
            if auth_headers(cfg) is None:                      # no login -> one clear failure, not hundreds
                download_and_ingest(state, cfg, cfg["url"], "-", "frost_obs.json")
            else:
                for url, month, filename in frost_observation_requests(state, cfg):
                    download_and_ingest(state, cfg, url, month, filename)

        manual = os.path.join(MANUAL_DIR, source, dataset)     # manual files for this dataset
        os.makedirs(manual, exist_ok=True)                     # create it so you can see where files go
        for name in sorted(os.listdir(manual)):                # every file in the manual folder
            if name.lower().endswith((".csv", ".json", ".zip")):  # only data files
                ingest_file(state, os.path.join(manual, name), source, dataset,
                            "manual", f"_manual/{source}/{dataset}", name)  # duplicates skipped by checksum

    shutil.rmtree(state["tmp_dir"], ignore_errors=True)        # empty the incoming folder of this run
    elapsed = time.time() - start_time
    logger.info(f"Bronze done in {elapsed:.1f}s: {state['counts']}")
    status = "success" if state["counts"]["failed"] == 0 else "failed"  # a failed REQUIRED source -> failed run
    return {"status": status, "elapsed_time": elapsed, **state["counts"]}

### 3.10 Run it

**Before the first run:**
1. Open the file `.env` in the repo root and replace `your-frost-client-id` with your Frost **client ID** (and the secret, for completeness).
   No `.env` yet? Copy `env.example` (repo root) to `.env`. The file is in `.gitignore`, so it is never pushed to GitHub.
2. Check the four `NORGESPRIS_…_MONTH` lines in `.env` (the period to fetch).
3. Re-run cell 3.2 after editing `.env` (or restart the kernel), so the new values are loaded.

The next cell prints the manual folders in `/tmp/norgespris-landing/<env>/_manual/` (only needed for manual files) and whether the Frost login was found.

Running the run cell twice is safe: unchanged files are skipped as `skipped_duplicate`, and only the 2 newest months of Frost data are requested again.
The first prod run makes a few hundred small Frost requests and downloads 3 years of Elhub data – expect it to take a while.

In [10]:
for cfg in SOURCES:                                            # show where each dataset's manual files go
    path = os.path.join(MANUAL_DIR, cfg["source"], cfg["dataset"])   # <landing>/_manual/<source>/<dataset>
    os.makedirs(path, exist_ok=True)                           # create the folder now
    print(f"{cfg['mode']:18s} {path}")                         # mode + folder
print("\nFrost login found:", auth_headers({"auth": "frost"}) is not None)  # True/False only, never the value
print(f"Period for {ENV}:", month_range())                    # (start, end) from .env; None = last complete month

snapshot           /tmp/norgespris-landing/dev/_manual/elhub/consumption_per_group_municipality_hour
snapshot           /tmp/norgespris-landing/dev/_manual/elhub/norgespris_consumption_per_group_eac_mba
snapshot           /tmp/norgespris-landing/dev/_manual/elhub/norgespris_count_per_municipality_consumption_group
snapshot           /tmp/norgespris-landing/dev/_manual/ssb/population_07459
snapshot           /tmp/norgespris-landing/dev/_manual/met/frost_stations
snapshot           /tmp/norgespris-landing/dev/_manual/met/frost_timeseries
frost_observations /tmp/norgespris-landing/dev/_manual/met/frost_observations
manual             /tmp/norgespris-landing/dev/_manual/met/seklima_daily_mean_temperature

Frost login found: True
Period for dev: ('2024-01', '2026-11')


In [11]:
result = run_bronze()                                          # run the whole Bronze layer
result                                                         # show the summary

2026-11-13 00:38:31,509 - bronze - INFO - Landing run 20261112T233831497455Z (env=dev) -> /tmp/norgespris-landing/dev
2026-11-13 00:38:31,510 - bronze - INFO - [elhub/consumption_per_group_municipality_hour] mode=snapshot
2026-11-13 00:38:46,513 - bronze - INFO -   skip (already ingested): consumption_per_group_municipality_hour snapshot consumption_per_group_municipality_hour.csv
2026-11-13 00:38:46,530 - bronze - INFO - [elhub/norgespris_consumption_per_group_eac_mba] mode=snapshot
2026-11-13 00:38:47,306 - bronze - INFO -   skip (already ingested): norgespris_consumption_per_group_eac_mba snapshot norgespris_consumption_per_group_eac_mba.csv
2026-11-13 00:38:47,308 - bronze - INFO - [elhub/norgespris_count_per_municipality_consumption_group] mode=snapshot
2026-11-13 00:38:47,882 - bronze - INFO -   unpacked gzip: norgespris_count_per_municipality_consumption_group.csv
2026-11-13 00:38:47,897 - bronze - INFO -   skip (already ingested): norgespris_count_per_municipality_consumption_g

{'status': 'success',
 'elapsed_time': 41.68411922454834,
 'ingested': 26,
 'skipped_duplicate': 4,
 'no_data': 0,
 'warning': 0,
 'failed': 0}

### 3.11 The manifest (traceability and monitoring)
Evidence for section 3 of the definition: where each file came from, when, its checksum, and whether it was new, a duplicate, empty, a schema change or a failure. The manifest is small, so pandas is enough here.

In [12]:
import pandas as pd                                            # small table -> pandas is fine

manifest = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("")   # read everything as text
cols = ["run_id", "dataset", "period", "original_filename", "status", "schema_status", "size_bytes"]
display(manifest[cols].tail(20))                               # newest 20 lines
print(manifest.groupby(["dataset", "status"]).size())          # files per dataset and status (all runs)

,run_id,dataset,period,original_filename,status,schema_status,size_bytes
679,20261112T233831497455Z,frost_observations,2026-10,frost_obs_2026-10_lcc5c39348438_g05.json,ingested,ok,1886125
680,20261112T233831497455Z,frost_observations,2026-10,frost_obs_2026-10_lcc5c39348438_g06.json,ingested,ok,1747674
681,20261112T233831497455Z,frost_observations,2026-10,frost_obs_2026-10_lcc5c39348438_g07.json,ingested,ok,1890169
682,20261112T233831497455Z,frost_observations,2026-10,frost_obs_2026-10_lcc5c39348438_g08.json,ingested,ok,1940182
683,20261112T233831497455Z,frost_observations,2026-10,frost_obs_2026-10_lcc5c39348438_g09.json,ingested,ok,1773719
684,20261112T233831497455Z,frost_observations,2026-10,frost_obs_2026-10_lcc5c39348438_g10.json,ingested,ok,1960041
685,20261112T233831497455Z,frost_observations,2026-10,frost_obs_2026-10_lcc5c39348438_g11.json,ingested,ok,2174203
686,20261112T233831497455Z,frost_observations,2026-10,frost_obs_2026-10_lcc5c39348438_g12.json,ingested,ok,271790
687,20261112T233831497455Z,frost_observations,2026-11,frost_obs_2026-11_lcc5c39348438_g01.json,ingested,ok,747046
688,20261112T233831497455Z,frost_observations,2026-11,frost_obs_2026-11_lcc5c39348438_g02.json,ingested,ok,747099


dataset                                              status                     
consumption_per_group_municipality_hour              ingested                         3
                                                     skipped_duplicate               15
frost_observations                                   ingested                       579
                                                     no_data                         12
frost_stations                                       ingested                        18
frost_timeseries                                     ingested                        18
norgespris_consumption_per_group_eac_mba             ingested                         3
                                                     skipped_duplicate               15
norgespris_count_per_municipality_consumption_group  ingested                         2
                                                     skipped_duplicate               15
population_07459                       

---
# 4. Check the landing files with Spark and profile the data (evidence for section 2)

This proves the landing files can be read by Spark and measures what Silver will have to fix.
CSV columns are read **as text**: Bronze/profiling must not guess types – type conversion is a Silver decision.

### 4.1 SparkSession and PostgreSQL settings
Small, explicit Spark settings: adaptive query execution on, 8 shuffle partitions (the default of 200 is far too many for this data size), 2 GB driver memory,
and `spark.sql.debug.maxToStringFields=200`: Spark describes each query plan as text for logs and the Spark UI and by default lists only 25 columns there –
our wide tables (Frost stations, the Norgespris file + 3 lineage columns) then trigger the harmless warning *"Truncated the string representation of a plan"*. Raising the limit only changes that log text, not the data.

The same session is used in section 5 to write to **PostgreSQL**, so it is created with the PostgreSQL **JDBC driver** (the Java library Spark uses to talk to Postgres):
* normally Spark downloads it from Maven Central the first time (`spark.jars.packages`, cached in `~/.ivy2`),
* without internet to Maven: download `postgresql-42.7.4.jar` (from jdbc.postgresql.org or the pgjdbc GitHub releases) and set `NORGESPRIS_PG_JDBC_JAR=/path/to/postgresql-42.7.4.jar` in `.env`.

The connection settings come from `.env` (`NORGESPRIS_PG_…`); the database name follows the environment: `norgespris_dev` / `norgespris_prod`.
⚠️ The driver is only loaded when the session **starts** – if a Spark session is already running in this kernel, restart the kernel first.

In [13]:
from pyspark.sql import SparkSession                            # entry point to Spark
from pyspark.sql import functions as F                          # column functions (count, min, max, ...)
from pyspark.sql import types as T                              # data types, to find nested JSON columns

PG_HOST = os.environ.get("NORGESPRIS_PG_HOST", "localhost")     # machine where PostgreSQL runs
PG_PORT = os.environ.get("NORGESPRIS_PG_PORT", "5432")          # PostgreSQL's standard port
PG_USER = os.environ.get("NORGESPRIS_PG_USER", "norgespris")    # the role created with scripts/init_postgres.sql
PG_PASSWORD = os.environ.get("NORGESPRIS_PG_PASSWORD", "")      # from .env or a GitHub secret - never in the code
PG_DATABASE = os.environ.get("NORGESPRIS_PG_DATABASE") or f"norgespris_{ENV}"  # norgespris_dev / norgespris_prod
PG_URL = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}" # JDBC address Spark connects to (no password in it)
PG_PROPERTIES = {"user": PG_USER, "password": PG_PASSWORD,      # login for every JDBC read/write
                 "driver": "org.postgresql.Driver"}             # Java class of the PostgreSQL driver
PG_JDBC_PACKAGE = "org.postgresql:postgresql:42.7.4"            # Maven coordinates of the driver
PG_JDBC_JAR = os.environ.get("NORGESPRIS_PG_JDBC_JAR", "").strip()  # optional: local .jar instead of Maven


def get_spark():
    """Start (or reuse) the SparkSession, with the PostgreSQL JDBC driver on the classpath."""
    builder = (SparkSession.builder
               .appName("DAT535-Group24-Bronze")                # name in the Spark UI
               .config("spark.sql.adaptive.enabled", "true")    # let Spark tune partitions/joins at runtime
               .config("spark.sql.shuffle.partitions", "8")     # small data -> few shuffle partitions
               .config("spark.driver.memory", "2g")             # enough for this data on a laptop
               .config("spark.sql.debug.maxToStringFields", "200"))  # wide tables: no "Truncated the string representation" warning
    if PG_JDBC_JAR:                                             # a downloaded driver file ...
        builder = builder.config("spark.jars", PG_JDBC_JAR)
    else:                                                       # ... or let Spark fetch it from Maven Central
        builder = builder.config("spark.jars.packages", PG_JDBC_PACKAGE)
    session = builder.getOrCreate()                             # reuse a running session if there is one
    session.sparkContext.setLogLevel("WARN")                    # hide Spark's INFO noise
    return session

In [14]:
spark = get_spark()                                             # one session for profiling (4) and loading (5)
print("Spark", spark.version, "| PostgreSQL target:", PG_URL)  # the URL holds no password

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/13 00:39:15 WARN Utils: Your hostname, X-sin-MacBook-Pro-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.4.120 instead (on interface en0)
26/11/13 00:39:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/x/UiS/DAT535%20Data%20Engineering/spark-env/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/x/.ivy2.5.2/cache
The jars for the packages stored in: /Users/x/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5c88e24e-e33d-4ca8-84c6-49d1d13270f9;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.4 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 114ms :: artifacts dl 7ms
	:: modules in use:
	org.

Spark 4.3.0 | PostgreSQL target: jdbc:postgresql://localhost:5432/norgespris_dev


### 4.2 Read the newest landing version of each dataset
* `newest_files` picks the newest version of **each file** (per period and file name) – for Frost it keeps only one station-group layout per month, avoiding a mix of old and new memberships.
* For a ZIP we read its unpacked folder (`…_unzipped/`); Spark reads every CSV in a folder and skips files starting with `_` or `.`.
* The separator is taken from the stored header line (more `;` than `,` → `;`), because the CSV inside Elhub's ZIP may differ from the files we downloaded by hand.
* We only remove the BOM character from the *column names* so we can refer to them – the data itself is not touched.
* **Encoding:** every file is read in the character set it is written in. `guess_encoding` checks the first 64 KB: valid UTF-8 → UTF-8,
  otherwise ISO-8859-1 (Latin-1). SSB's API writes Latin-1 (`å` = one byte); reading it as UTF-8 would turn `Råde` into `R�de` in PostgreSQL – lost for good.
  Files with different encodings in one dataset are read separately and combined (`unionByName`).
* JSON datasets (Frost) are read separately in 4.5.
* Every row gets the column **`_source_file`** (the landing file it came from, via Spark's hidden `_metadata` column) – this is the lineage we store in PostgreSQL in section 5.

In [15]:
def newest_files(dataset):
    """Absolute paths of the newest ingested version of every file of a dataset."""
    all_rows = manifest[manifest.dataset == dataset]
    if dataset == "frost_observations" and not all_rows.empty:
        all_rows = all_rows.copy()
        all_rows["_layout"] = all_rows.original_filename.str.extract(r"_l([0-9a-f]{12})_g\d+\.json$")[0].fillna("legacy")
        layouts = all_rows[all_rows.status.isin(("ingested", "skipped_duplicate", "no_data"))]
        latest_layout = layouts.sort_values("run_id").groupby("period").tail(1)[["period", "_layout"]]
        all_rows = all_rows.merge(latest_layout, on=["period", "_layout"])
    rows = all_rows[all_rows.status.isin(("ingested", "skipped_duplicate"))
                    & all_rows.landing_path.notna()]  # latest successful fetches with a stored path
    newest = rows.sort_values("run_id").groupby(["period", "original_filename"]).tail(1)  # newest per file
    paths = [os.path.join(LANDING_DIR, p) for p in newest.landing_path]                 # full paths
    return [p[:-len(".zip")] + "_unzipped" if p.endswith(".zip") else p for p in paths]  # ZIP -> its unpacked folder

def guess_sep(dataset, default):
    """Separator from the header line stored in the manifest: ';' if it has more ';' than ','."""
    rows = manifest[(manifest.dataset == dataset) & manifest.status.isin(("ingested", "skipped_duplicate"))]
    if rows.empty:                                              # nothing stored -> use the registry value
        return default
    header = rows.sort_values("run_id").header.iloc[-1]         # header of the newest file
    return ";" if header.count(";") > header.count(",") else ","

def guess_encoding(path):
    """'UTF-8' if the start of the file is valid UTF-8, otherwise 'ISO-8859-1' (Latin-1, e.g. SSB's API: 'å' = one byte)."""
    if os.path.isdir(path):                                     # unpacked ZIP folder -> look at its first CSV
        csvs = sorted(f for f in os.listdir(path) if f.lower().endswith(".csv"))
        if not csvs:
            return "UTF-8"
        path = os.path.join(path, csvs[0])
    with open(path, "rb") as f:
        sample = f.read(65536)                                  # the first 64 KB are enough to decide
    try:
        sample.decode("utf-8")                                  # valid UTF-8 (plain ASCII is too) -> UTF-8
        return "UTF-8"
    except UnicodeDecodeError as e:
        if e.start >= len(sample) - 3:                          # only the last character was cut by the 64 KB limit
            return "UTF-8"
        return "ISO-8859-1"                                     # bytes that are not UTF-8 -> Latin-1 (Norwegian letters = 1 byte)

def read_bronze(cfg):
    """Read a Bronze CSV dataset with Spark, all columns as text; None if nothing to read."""
    files = newest_files(cfg["dataset"])                        # which files/folders to read
    if not files or cfg["sep"] is None:                         # nothing ingested, or JSON (Frost -> 4.5)
        return None
    sep = guess_sep(cfg["dataset"], cfg["sep"])                 # Elhub's ZIP may use another separator than our samples
    by_encoding = {}                                            # {'UTF-8': [files], 'ISO-8859-1': [files]}
    for path in files:
        by_encoding.setdefault(guess_encoding(path), []).append(path)
    parts = []
    for encoding, group in by_encoding.items():                 # one Spark read per encoding
        df = (spark.read
              .option("header", True)                           # first line = column names
              .option("inferSchema", False)                     # keep everything as text (raw)
              .option("sep", sep)                               # separator for this dataset
              .option("encoding", encoding)                     # the encoding the files are written in
              .csv(group))                                      # list of files -> one DataFrame
        clean = [c.replace("\ufeff", "").strip() for c in df.columns]  # remove BOM from column names only
        parts.append(df.select(*[F.col(f"`{c}`").alias(n) for c, n in zip(df.columns, clean)],  # same columns, clean names
                               F.col("_metadata.file_path").alias("_source_file")))         # + which landing file
    result = parts[0]
    for part in parts[1:]:                                      # files with different encodings -> one DataFrame
        result = result.unionByName(part, allowMissingColumns=True)  # columns only in some files -> NULL elsewhere
    return result

In [16]:
bronze = {cfg["dataset"]: read_bronze(cfg) for cfg in SOURCES}  # one DataFrame per CSV dataset (None for JSON)

### 4.3 Row counts and columns
`count()` is an *action*: this is where Spark actually reads the files. These numbers go into the report as the Bronze row counts that Silver results are compared with.

In [17]:
for name, df in bronze.items():                                 # every CSV dataset
    if df is None:                                              # skip what is not ingested / JSON
        continue
    print(f"\n=== {name}: {df.count():,} rows, {len(df.columns)} columns")  # action -> Spark reads the files
    print(df.columns)                                           # column names
    df.show(3, truncate=False)                                  # first 3 rows


=== consumption_per_group_municipality_hour: 28,250,496 rows, 9 columns
['START_TIME', 'END_TIME', 'MUNICIPALITY_NUMBER', 'NAME', 'CONSUMPTION_GROUP', 'QUANTITY_KWH', 'METERING_POINT_COUNT', 'LAST_UPDATE_TIME', '_source_file']
+-----------------------------+-----------------------------+-------------------+------------------+-----------------+------------+--------------------+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|START_TIME                   |END_TIME                     |MUNICIPALITY_NUMBER|NAME              |CONSUMPTION_GROUP|QUANTITY_KWH|METERING_POINT_COUNT|LAST_UPDATE_TIME             |_source_file                                                                                                                                                                                 

### 4.4 Source-specific checks that Silver must handle
One small query per known risk. Each printed number is a finding for the report.

In [18]:
cons = bronze["consumption_per_group_municipality_hour"]        # Elhub hourly consumption
if cons is not None:
    cons.agg(F.min("START_TIME").alias("first_hour"),           # period covered by the files
             F.max("START_TIME").alias("last_hour"),
             F.countDistinct("MUNICIPALITY_NUMBER").alias("municipalities")).show(truncate=False)
    cons.groupBy("CONSUMPTION_GROUP").count().show()            # rows per consumption group
    print("Rows with municipality 0 ('unknown'):",
          cons.filter(F.col("MUNICIPALITY_NUMBER") == "0").count())       # cannot be joined to SSB -> handle in Silver
    print("Names with broken letters (√ instead of æøå):",
          cons.filter(F.col("NAME").contains("√")).select("NAME").distinct().count())  # encoding check
    print("Municipality numbers shorter than 4 digits:",
          cons.filter(F.length("MUNICIPALITY_NUMBER") < 4).select("MUNICIPALITY_NUMBER").distinct().count())

+-----------------------------+-----------------------------+--------------+
|first_hour                   |last_hour                    |municipalities|
+-----------------------------+-----------------------------+--------------+
|2023-11-12T00:00:00.000+01:00|2026-11-11T23:00:00.000+01:00|358           |
+-----------------------------+-----------------------------+--------------+



+-----------------+-------+
|CONSUMPTION_GROUP|  count|
+-----------------+-------+
|          private|9416832|
|         industry|9416832|
|         business|9416832|
+-----------------+-------+



Rows with municipality 0 ('unknown'): 0


Names with broken letters (√ instead of æøå): 0


Municipality numbers shorter than 4 digits: 0


In [19]:
pop = bronze["population_07459"]                                # SSB population
if pop is not None:
    region = pop.columns[0]                                     # region column ("Kommune"/"region")
    pop.select(region).show(3, truncate=False)                  # codes look like "K-3101 Halden"
    print("Rows that are not real municipalities:",
          pop.filter(F.col(region).rlike("Rest|Kontinentalsokkelen|[Uu]oppgitt")).count())  # remove in Silver

+-------------------------+
|Region - region          |
+-------------------------+
|K-3101 - K-3101 Halden   |
|K-3103 - K-3103 Moss     |
|K-3105 - K-3105 Sarpsborg|
+-------------------------+
only showing top 3 rows
Rows that are not real municipalities: 2


### 4.5 Frost: stations, observations – and a preview of mean temperature per municipality

Frost answers in JSON with the records inside a list called `data`. Spark reads JSON directly
(`multiLine=True` because each file is one big JSON document, not one record per line), and `explode` turns the list into rows.

* **Stations:** one row per station with `municipalityId` (a number like `301` → we write it as `0301` with `lpad`, the same 4-digit form Elhub uses).
* **Observations:** each record is one station and one day, with a list of observations inside – usually one, but a station can have **several daily-mean series** with different `timeOffset` (when the 24-hour period starts, e.g. `PT0H` = midnight UTC, `PT6H` = 06 UTC). We count them first.
* **Preview (Silver logic, not saved):** keep one series per station and day (`PT0H` if it exists, otherwise the first one), join with the stations, and average per municipality and day.
  This shows that the data supports our question; the real, tested version comes in the Silver notebook.

In [20]:
def read_frost(dataset):
    """Read all newest Frost JSON files of a dataset with Spark and explode the 'data' list into rows."""
    files = newest_files(dataset)                               # all JSON files of the dataset
    if not files:
        print(f"-- {dataset}: nothing in landing yet (is FROST_CLIENT_ID set?)")
        return None
    raw = spark.read.option("multiLine", True).json(files)      # one JSON document per file
    return (raw.select(F.explode("data").alias("d"),            # one row per element of "data"
                       F.col("_metadata.file_path").alias("_source_file"))  # + which landing file
               .select("d.*", "_source_file"))

In [21]:
stations = read_frost("frost_stations")                         # step 1: station -> municipality
observations = read_frost("frost_observations")                 # step 3: daily means

if stations is not None:
    stations = stations.select(
        F.col("id").alias("station_id"),                        # e.g. SN18700
        F.col("name").alias("station_name"),
        F.lpad(F.col("municipalityId").cast("string"), 4, "0").alias("municipality_no"),  # 301 -> "0301"
        F.col("municipality").alias("municipality_name"),
        F.col("county").alias("county_name"))
    print("Stations:", stations.count(),                       # all stations in the period
          "| with a municipality:", stations.filter(F.col("municipality_no").isNotNull()).count())
    stations.show(3, truncate=False)

Stations: 2307 | with a municipality: 1911
+----------+-----------------+---------------+-----------------+-----------+
|station_id|station_name     |municipality_no|municipality_name|county_name|
+----------+-----------------+---------------+-----------------+-----------+
|SN47230   |ÅKRA UNGDOMSSKOLE|1149           |KARMØY           |ROGALAND   |
|SN20952   |STATFJORD C      |2600           |NORDSJØEN        |NORDSJØEN  |
|SN23670   |E16 RYFOSS       |3454           |VANG             |INNLANDET  |
+----------+-----------------+---------------+-----------------+-----------+
only showing top 3 rows


In [22]:
if observations is not None:
    obs = (observations
           .select(F.split("sourceId", ":")[0].alias("station_id"),             # "SN18700:0" -> "SN18700"
                   F.to_date(F.substring("referenceTime", 1, 10)).alias("date"),  # "2025-10-01T00:00:00.000Z" -> date
                   F.explode("observations").alias("o"))                         # one row per observation
           .select("station_id", "date",
                   F.col("o.value").cast("double").alias("mean_temp_c"),         # the temperature in °C
                   F.col("o.timeOffset").alias("time_offset")))                 # which daily-mean series
    print("Observation rows:", obs.count())
    obs.groupBy("time_offset").count().orderBy("time_offset").show()             # how many series per offset
    obs.agg(F.min("date").alias("first_day"), F.max("date").alias("last_day"),
            F.countDistinct("station_id").alias("stations")).show()

Observation rows: 1494436


+-----------+------+
|time_offset| count|
+-----------+------+
|       PT0H|913828|
|       PT6H|580608|
+-----------+------+



+----------+----------+--------+
| first_day|  last_day|stations|
+----------+----------+--------+
|2024-01-01|2026-11-12|    1110|
+----------+----------+--------+



In [23]:
if stations is not None and observations is not None:
    from pyspark.sql import Window                               # to pick one series per station and day

    pick = Window.partitionBy("station_id", "date").orderBy(    # within one station and day ...
        F.when(F.col("time_offset") == "PT0H", 0).otherwise(1),  # ... prefer the PT0H series
        "time_offset")                                          # ... otherwise the first offset
    one_per_day = (obs.withColumn("rank", F.row_number().over(pick))
                      .filter("rank = 1").drop("rank"))         # exactly one value per station and day

    muni_temp = (one_per_day
                 .join(F.broadcast(stations), "station_id")     # small station table -> broadcast join (no shuffle of obs)
                 .filter(F.col("municipality_no").isNotNull())  # stations outside a municipality (sea, abroad)
                 .groupBy("municipality_no", "municipality_name", "date")
                 .agg(F.round(F.avg("mean_temp_c"), 1).alias("mean_temp_c"),     # average of all stations
                      F.count("*").alias("n_stations")))                        # how many stations it is based on

    muni_temp.orderBy("municipality_no", "date").show(10, truncate=False)
    print("Municipalities with at least one station:", muni_temp.select("municipality_no").distinct().count(),
          "of ~357 -> the rest need a fallback in Silver (county average or nearest station)")

+---------------+-----------------+----------+-----------+----------+
|municipality_no|municipality_name|date      |mean_temp_c|n_stations|
+---------------+-----------------+----------+-----------+----------+
|0301           |OSLO             |2024-01-01|-5.4       |12        |
|0301           |OSLO             |2024-01-02|-8.0       |12        |
|0301           |OSLO             |2024-01-03|-9.2       |12        |
|0301           |OSLO             |2024-01-04|-11.4      |12        |
|0301           |OSLO             |2024-01-05|-17.9      |12        |
|0301           |OSLO             |2024-01-06|-18.7      |12        |
|0301           |OSLO             |2024-01-07|-16.8      |12        |
|0301           |OSLO             |2024-01-08|-14.7      |12        |
|0301           |OSLO             |2024-01-09|-8.5       |12        |
|0301           |OSLO             |2024-01-10|-6.1       |11        |
+---------------+-----------------+----------+-----------+----------+
only showing top 10 

Municipalities with at least one station: 321 of ~357 -> the rest need a fallback in Silver (county average or nearest station)


---
# 5. Bronze layer in PostgreSQL

The **Bronze layer is the `bronze` schema in PostgreSQL**: the raw landing files, loaded as they are (all text) into one table per dataset.
Every layer ends up as tables in PostgreSQL (`bronze` now, `silver` and `gold` later), so the dashboard,
SQL queries and the examiner can use the data without reading files.

| Choice | Why |
|---|---|
| Landing files → Bronze **tables** | The landing files are short-lived (`/tmp`); the Bronze tables keep the raw content, queryable with SQL. ZIPs are read from their unpacked copy, JSON documents are split into one row per record. |
| One database per environment: `norgespris_dev`, `norgespris_prod` | Same separation as the landing folders – a dev run never touches prod tables. |
| Schemas `bronze`, `silver`, `gold` (created by `scripts/init_postgres.sql`) | The Medallion layers are visible in the database. |
| One table per dataset: `bronze.<dataset>` | Same names as the landing folders – easy to trace. |
| **All columns as text** (+ nested JSON as JSON text) | Bronze must not interpret data; types are a Silver decision. A JDBC column cannot hold a nested structure, so Frost's lists/objects are stored as JSON text. |
| Lineage columns `_source_file`, `_landing_run_id`, `_loaded_at` | Every row can be traced back to the exact landing file and run. |
| Written by **Spark over JDBC** (`DataFrame.write.jdbc`) | Same tool as the rest of the pipeline; Spark writes in parallel and in batches (`batchsize=10000`). |
| **Reload only when the landing files changed** (`bronze._load_log`) | The Elhub table has tens of millions of rows. A fingerprint of the newest landing files is compared with the last load; unchanged → skipped. Changed → the table is replaced (`overwrite`), so a rerun never creates duplicates. |
| The manifest is copied to `bronze.manifest` | Monitoring with SQL: what was ingested when, what failed. |
| Password from `.env` / GitHub secret | Never in code or notebook output. |

### 5.1 Helper functions

In [24]:
import hashlib                                                  # fingerprint of the landing files of a dataset
import time                                                     # elapsed time
from datetime import datetime, timezone                         # load timestamps (UTC)

import pandas as pd                                             # the manifest is a small table

pg_logger = logging.getLogger("postgres")                       # own logger name in the run log


def to_postgres_types(df):
    """Nested columns (struct/array/map, e.g. Frost's lists) -> JSON text; other columns unchanged."""
    cols = []
    for field in df.schema.fields:                              # look at every column's data type
        name = f"`{field.name}`"                                # backticks: names may contain spaces/dots
        if isinstance(field.dataType, (T.StructType, T.ArrayType, T.MapType)):
            cols.append(F.to_json(F.col(name)).alias(field.name))  # nested -> JSON text
        else:
            cols.append(F.col(name).cast("string").alias(field.name))  # everything else as text
    return df.select(*cols)


def add_lineage(df):
    """Add _landing_run_id (from the file name '<run_id>__...') and _loaded_at to every row."""
    return (df.withColumn("_landing_run_id",
                          F.regexp_extract("_source_file", r"/(\d{8}T\d+Z)__", 1))  # run that stored the file
              .withColumn("_loaded_at", F.current_timestamp()))             # when it was written to Postgres


def bronze_fingerprint(dataset):
    """Short hash of the newest landing file paths of a dataset (paths contain the run_id) - None if no files."""
    files = newest_files(dataset)                               # same file selection as the profiling
    if not files:
        return None
    return hashlib.sha256("\n".join(sorted(files)).encode("utf-8")).hexdigest()[:16]


def pg_query(sql):
    """Run a SELECT in PostgreSQL through Spark and return the result as a Spark DataFrame."""
    return spark.read.jdbc(PG_URL, f"({sql}) AS q", properties=PG_PROPERTIES)  # the query runs inside Postgres


def write_table(df, table, mode="overwrite"):
    """Write a DataFrame to a PostgreSQL table (overwrite = drop and recreate, append = add rows)."""
    (df.write
       .option("batchsize", 10000)                              # rows per INSERT batch - much faster than 1 by 1
       .jdbc(PG_URL, table, mode=mode, properties=PG_PROPERTIES))


def read_load_log():
    """Last fingerprint per dataset from bronze._load_log ({} on the very first load)."""
    try:
        log = pg_query("SELECT dataset, fingerprint, loaded_at FROM bronze._load_log").toPandas()
    except Exception as e:                                      # the table does not exist before the first load
        if "does not exist" in str(e):
            return {}
        raise                                                   # other errors (wrong password, no server) -> stop
    newest = log.sort_values("loaded_at").groupby("dataset").tail(1)   # newest load per dataset
    return dict(zip(newest.dataset, newest.fingerprint))

### 5.2 Load every dataset
For each dataset in the source registry: compute the fingerprint → skip if unchanged → otherwise read it with the functions from section 4
(`read_bronze` for CSV, `read_frost` for JSON) → convert types → add lineage → write `bronze.<dataset>` → count the rows **in Postgres**
(the count runs as SQL inside the database, so we prove what actually arrived).

A failing **required** dataset makes the stage fail (red in GitHub Actions); datasets without landing files are just skipped.

In [25]:
def load_dataset(cfg, last_fingerprints):
    """Load one dataset from landing into bronze.<dataset>; returns a dict for the load log."""
    dataset = cfg["dataset"]
    table = f"bronze.{dataset}"
    result = {"dataset": dataset, "table": table, "status": "", "fingerprint": "", "rows": 0}
    fingerprint = bronze_fingerprint(dataset)                   # which landing files would be loaded
    if fingerprint is None:                                     # nothing in landing (e.g. optional source)
        result["status"] = "no_landing_files"
        return result
    result["fingerprint"] = fingerprint
    if last_fingerprints.get(dataset) == fingerprint:           # same files as last time
        result["status"] = "unchanged"
        return result
    df = read_bronze(cfg) if cfg["sep"] is not None else read_frost(dataset)  # CSV or JSON reader from section 4
    df = add_lineage(to_postgres_types(df))                     # text columns + lineage
    write_table(df, table)                                      # replace the table
    result["rows"] = pg_query(f"SELECT COUNT(*) AS n FROM {table}").first()["n"]  # rows that arrived in Postgres
    result["status"] = "loaded"
    pg_logger.info(f"  {table}: {result['rows']:,} rows")
    return result


def run_bronze_to_postgres():
    """Load the newest landing files and the manifest into the bronze schema in PostgreSQL; returns a summary."""
    global spark, manifest                                      # used by the reader functions from section 4
    start_time = time.time()
    if not PG_PASSWORD or PG_PASSWORD.startswith("your-"):      # missing or still the placeholder from env.example
        pg_logger.error("NORGESPRIS_PG_PASSWORD is not set (in .env or as GitHub secret)")  # clear message, no stack trace
        return {"status": "failed", "elapsed_time": 0.0}
    spark = get_spark()
    manifest = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("") # newest manifest (landing may just have run)
    pg_logger.info(f"Loading landing files into the bronze schema of {PG_URL}")
    last = read_load_log()                                      # what was loaded last time
    loaded_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    results, failed = [], 0
    for cfg in SOURCES:                                         # same order as the registry
        try:
            results.append(load_dataset(cfg, last))
        except Exception as e:                                  # keep going, but remember the failure
            pg_logger.error(f"  bronze.{cfg['dataset']}: {type(e).__name__}: {str(e)[:300]}")
            results.append({"dataset": cfg["dataset"], "table": f"bronze.{cfg['dataset']}",
                            "status": f"failed: {type(e).__name__}", "fingerprint": "", "rows": 0})
            failed += 1 if cfg["required"] else 0               # optional datasets do not fail the stage

    write_table(spark.createDataFrame(manifest), "bronze.manifest")  # the manifest as a table (replaced each run)
    done = [dict(r, loaded_at=loaded_at) for r in results if r["status"] == "loaded"]
    if done:                                                    # remember what was loaded (append = history)
        write_table(spark.createDataFrame(pd.DataFrame(done)), "bronze._load_log", mode="append")

    elapsed = time.time() - start_time
    counts = pd.Series([r["status"].split(":")[0] for r in results]).value_counts().to_dict()
    pg_logger.info(f"PostgreSQL load done in {elapsed:.1f}s: {counts}")
    return {"status": "success" if failed == 0 else "failed", "elapsed_time": elapsed,
            "tables": results, **counts}

### 5.3 Run it
Needs a running PostgreSQL with the databases from `scripts/init_postgres.sql` and `NORGESPRIS_PG_PASSWORD` in `.env` (see README).
The first prod load of the Elhub table takes a while (tens of millions of rows); later runs skip unchanged datasets.

In [26]:
pg_result = run_bronze_to_postgres()                            # load all datasets
pd.DataFrame(pg_result.get("tables", []))                       # one line per table: status and rows

2026-11-13 00:41:20,639 - postgres - INFO - Loading landing files into the bronze schema of jdbc:postgresql://localhost:5432/norgespris_dev
/Users/x/UiS/DAT535 Data Engineering/spark-env/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
2026-11-13 00:49:01,431 - postgres - INFO -   bronze.consumption_per_group_municipality_hour: 28,250,496 rows
2026-11-13 00:49:03,260 - postgres - INFO -   bronze.norgespris_consumption_per_group_eac_mba: 43,712 rows
2026-11-13 00:49:06,926 - postgres - INFO -   bronze.norgespris_count_per_municipality_consumption_group: 289,645 rows
2026-11-13 00:49:07,253 - postgres - INFO -   bronze.population_07459: 360 rows
2026-11-13 00:49:09,190 - postgres - INFO -   bronze.frost_stations: 2,307 rows
2026-11-13 00:49:09,713 - postgres - INFO -   bronze.frost_ti

,dataset,table,status,fingerprint,rows
0,consumption_per_group_municipality_hour,bronze.consumption_per_group_municipality_hour,loaded,5eef0791682478cd,28250496
1,norgespris_consumption_per_group_eac_mba,bronze.norgespris_consumption_per_group_eac_mba,loaded,06ffb4612329f045,43712
2,norgespris_count_per_municipality_consumption_...,bronze.norgespris_count_per_municipality_consu...,loaded,89c058fe1c8bf05c,289645
3,population_07459,bronze.population_07459,loaded,ec4a19e430965076,360
4,frost_stations,bronze.frost_stations,loaded,17541b6a5837b165,2307
5,frost_timeseries,bronze.frost_timeseries,loaded,c867f9d0af44b3d7,1996
6,frost_observations,bronze.frost_observations,loaded,5a8580e585ad62d9,913946
7,seklima_daily_mean_temperature,bronze.seklima_daily_mean_temperature,no_landing_files,,0


### 5.4 Check the database
The same check you can do in `psql` (`\dt bronze.*` and `SELECT * FROM bronze._load_log;`), but through Spark:
which tables exist in the `bronze` schema, their size, and the load history.

In [27]:
pg_query("""SELECT table_name,
                   pg_size_pretty(pg_total_relation_size('bronze.' || quote_ident(table_name))) AS size
            FROM information_schema.tables
            WHERE table_schema = 'bronze'
            ORDER BY table_name""").show(truncate=False)        # tables in the bronze schema
pg_query("SELECT dataset, rows, loaded_at, fingerprint FROM bronze._load_log "
         "ORDER BY loaded_at DESC").show(10, truncate=False)    # newest loads first

+---------------------------------------------------+-------+
|table_name                                         |size   |
+---------------------------------------------------+-------+
|_load_log                                          |16 kB  |
|consumption_per_group_municipality_hour            |11 GB  |
|frost_observations                                 |575 MB |
|frost_stations                                     |976 kB |
|frost_timeseries                                   |1272 kB|
|manifest                                           |840 kB |
|norgespris_consumption_per_group_eac_mba           |22 MB  |
|norgespris_count_per_municipality_consumption_group|84 MB  |
|population_07459                                   |112 kB |
+---------------------------------------------------+-------+

+---------------------------------------------------+--------+-------------------------+----------------+
|dataset                                            |rows    |loaded_at                

---
# 6. How to run (summary)

| Where | Command |
|---|---|
| Notebook (dev) | Fill in `.env`, run all cells; raw files in `/tmp/norgespris-landing/dev/`, tables in `norgespris_dev` |
| Terminal | `bash scripts/setup_env.sh` once, then `source ~/spark-env/bin/activate && python run_pipeline.py all` (= `bronze`: sources → landing, then `postgres`: landing → `bronze` schema) |
| Tests | `pytest -q tests/` |
| GitHub Actions – dev | push to `dev` (or run by hand) → `norgespris-dev.yml`: tests, then the pipeline (landing `/tmp/norgespris-landing/dev`, database `norgespris_dev`) |
| GitHub Actions – prod | merge to `main` (+ approval) or run by hand → `norgespris-prod.yml` |
| GitHub Actions – schedule | 5th of every month → `pipeline.yml` refreshes prod |

GitHub Actions does not see your `.env` (it is not in git):
* the client ID goes in as repository **secret** `FROST_CLIENT_ID`, the database password as secret `NORGESPRIS_PG_PASSWORD`,
* the period goes in as repository **variables** `NORGESPRIS_DEV_START_MONTH`, … (*Settings → Secrets and variables → Actions → Variables*).
  If a variable is not set, the workflow file uses the same default values as `env.example`.

The machine needs internet access to `api.elhub.no`, `data.elhub.no`, `data.ssb.no` and `frost.met.no`, and a running PostgreSQL (see README).

**Next notebook (02):** Silver – read the newest Bronze data with explicit schemas, clean, standardise and write `silver.*` tables in PostgreSQL.